In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from torch.nn import functional as F
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
import seaborn as sns

In [2]:
# 自定義數據集
class SegmentationDataset(Dataset):
    def __init__(self, image_dir, mask_dir, is_train=True):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.is_train = is_train
        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))
        
        # 基本轉換
        self.transform = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),  # 轉換為灰度圖
            transforms.ToTensor(),
        ])
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])
        
        # 讀取圖像
        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')
        
        # 調整圖像大小到統一尺寸
        target_size = (512, 512)  # 可以根據需求調整
        image = image.resize(target_size, Image.BILINEAR)
        mask = mask.resize(target_size, Image.NEAREST)
        
        # 轉換
        image = self.transform(image)
        mask = torch.from_numpy(np.array(mask))
        mask = (mask > 128).float()  # 二值化
        
        return image, mask

# 定義 U-Net 模型
class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()
        
        # Encoder
        self.enc1 = self._double_conv(1, 64)
        self.enc2 = self._double_conv(64, 128)
        self.enc3 = self._double_conv(128, 256)
        self.enc4 = self._double_conv(256, 512)
        
        # Decoder
        self.dec4 = self._double_conv(512 + 256, 256)
        self.dec3 = self._double_conv(256 + 128, 128)
        self.dec2 = self._double_conv(128 + 64, 64)
        self.dec1 = nn.Conv2d(64, 1, kernel_size=1)
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
    def _double_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        # Decoder
        d4 = self.dec4(torch.cat([self.upsample(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.upsample(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.upsample(d3), e1], dim=1))
        
        return torch.sigmoid(self.dec1(d2))

In [3]:
def calculate_metrics(pred, target):
    """計算 IoU 和 F1 score"""
    pred = pred > 0.5
    target = target > 0.5
    
    # 計算 IoU
    intersection = np.logical_and(target, pred)
    union = np.logical_or(target, pred)
    iou = np.sum(intersection) / (np.sum(union) + 1e-10)
    
    # 計算 F1
    f1 = f1_score(target.flatten(), pred.flatten())
    
    return iou, f1

def plot_loss_curve(losses, save_path='loss_curve.png'):
    """繪製損失曲線"""
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Training Loss')
    plt.title('Training Loss Over Time')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()


In [4]:
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=50):
    """訓練模型並記錄損失"""
    losses = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        batch_count = 0
        
        for i, (images, masks) in enumerate(train_loader):
            images = images.to(device)
            masks = masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks.unsqueeze(1))
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            batch_count += 1
            
            # 打印每個batch的進度
            print(f'Epoch {epoch+1}/{num_epochs}, Batch {i}/{len(train_loader)-1}, Loss: {loss.item():.4f}')
        
        # 確保使用正確的batch數量計算平均損失
        avg_loss = epoch_loss / batch_count
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}')
        print(f'Processed {batch_count} batches in this epoch')
        
        # 保存模型
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'unet_model_epoch_{epoch+1}.pth')
    
    # 繪製損失曲線
    plot_loss_curve(losses)
    
    return losses

def visualize_results(original_rgb, prediction, ground_truth, image_name, save_path):
    """生成包含原圖、預測結果和真實標籤的對比圖"""
    plt.figure(figsize=(15, 5))
    
    # 設置字體大小
    plt.rcParams.update({'font.size': 10})
    
    # 原始彩色圖像
    plt.subplot(131)
    plt.title(f'Original\n{image_name}', pad=10)
    plt.imshow(original_rgb)
    plt.axis('off')
    
    # 預測結果
    plt.subplot(132)
    plt.title('Prediction', pad=10)
    plt.imshow(prediction, cmap='gray') # 灰度圖 (0-1)
    plt.axis('off')
    
    # 真實標籤
    plt.subplot(133)
    plt.title('Ground Truth', pad=10)
    plt.imshow(ground_truth, cmap='gray')
    plt.axis('off')
    
    plt.tight_layout(pad=3.0)  # 增加間距
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

def test_model(model, test_dir, results_dir, device):
    """測試模型並生成結果"""
    model.eval()
    
    # 創建結果目錄
    os.makedirs(results_dir, exist_ok=True)
    results = []
    
    # 設置轉換
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ])
    
    # 處理編號從01到19的圖像    # 寫死了，改天再改
    for i in range(1, 20):
        input_name = f"{i:02d}_real_A.png"
        gt_name = f"{i:02d}_real_B.png"
        
        img_path = os.path.join(test_dir, input_name)
        gt_path = os.path.join(test_dir, gt_name)
        
        if not os.path.exists(img_path) or not os.path.exists(gt_path):
            print(f"Warning: Missing files for image {i}")
            continue
        
        # 讀取原始彩色圖像
        original_rgb = Image.open(img_path).convert('RGB')
        original_size = original_rgb.size
        
        # 處理輸入圖像（轉換為灰度用於模型）
        input_tensor = transform(original_rgb)
        input_tensor = input_tensor.unsqueeze(0).to(device)
        input_tensor = F.interpolate(input_tensor, size=(512, 512), mode='bilinear', align_corners=True)
        
        # 預測
        with torch.no_grad():
            output = model(input_tensor)
        
        # 將預測結果轉換回原始大小
        output = F.interpolate(output, size=original_size, mode='bilinear', align_corners=True)
        prediction = output.squeeze().cpu().numpy()
        
        # 讀取真實標籤
        ground_truth = np.array(Image.open(gt_path).convert('L'))
        
        # 計算評估指標
        gt_binary = ground_truth > 128
        pred_binary = prediction > 0.5
        iou, f1 = calculate_metrics(pred_binary, gt_binary)
        
        # 保存結果
        results.append({
            'Image': f'Image_{i:02d}',
            'IoU': iou,
            'F1': f1
        })
        
        # 生成視覺化結果
        save_path = os.path.join(results_dir, f'result_{i:02d}.png')
        visualize_results(
            np.array(original_rgb),
            prediction,
            ground_truth,
            f'Image {i:02d}',
            save_path
        )
    
    # 創建DataFrame並保存為CSV
    df = pd.DataFrame(results)
    
    # 添加平均值行
    avg_row = {
        'Image': 'Average',
        'IoU': df['IoU'].mean(),
        'F1': df['F1'].mean()
    }
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    
    csv_path = os.path.join(results_dir, 'metrics.csv')
    df.to_csv(csv_path, index=False)
    
    # 打印結果摘要
    print("\nTest Results Summary:")
    print(df.to_string(index=False))
    
    return df


In [5]:
# 設置設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 創建數據加載器
train_dataset = SegmentationDataset(
    image_dir='DRIVE/training/images',
    mask_dir='DRIVE/training/1st_manual',
    is_train=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Total number of training samples: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(train_loader)}")

# 初始化模型
model = UNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 訓練模型
losses = train_model(model, train_loader, criterion, optimizer, device, num_epochs=100)

Using device: cuda
Total number of training samples: 20
Number of batches per epoch: 5
Epoch 1/100, Batch 0/4, Loss: 0.7011
Epoch 1/100, Batch 1/4, Loss: 0.6553
Epoch 1/100, Batch 2/4, Loss: 0.6016
Epoch 1/100, Batch 3/4, Loss: 0.5511
Epoch 1/100, Batch 4/4, Loss: 0.4976
Epoch 1/100, Average Loss: 0.6013
Processed 5 batches in this epoch
Epoch 2/100, Batch 0/4, Loss: 0.4582
Epoch 2/100, Batch 1/4, Loss: 0.4328
Epoch 2/100, Batch 2/4, Loss: 0.4111
Epoch 2/100, Batch 3/4, Loss: 0.3928
Epoch 2/100, Batch 4/4, Loss: 0.3905
Epoch 2/100, Average Loss: 0.4171
Processed 5 batches in this epoch
Epoch 3/100, Batch 0/4, Loss: 0.3654
Epoch 3/100, Batch 1/4, Loss: 0.3591
Epoch 3/100, Batch 2/4, Loss: 0.3461
Epoch 3/100, Batch 3/4, Loss: 0.3608
Epoch 3/100, Batch 4/4, Loss: 0.3403
Epoch 3/100, Average Loss: 0.3543
Processed 5 batches in this epoch
Epoch 4/100, Batch 0/4, Loss: 0.3457
Epoch 4/100, Batch 1/4, Loss: 0.3210
Epoch 4/100, Batch 2/4, Loss: 0.3189
Epoch 4/100, Batch 3/4, Loss: 0.3171
Epoch 

In [6]:
# 測試模型
results_df = test_model(
    model,
    test_dir='DIRVE_TestingSet',
    results_dir='test_results',
    device=device
)


Test Results Summary:
   Image      IoU       F1
Image_01 0.668726 0.801481
Image_02 0.707754 0.828871
Image_03 0.585108 0.738256
Image_04 0.674457 0.805583
Image_05 0.639795 0.780336
Image_06 0.627565 0.771170
Image_07 0.641031 0.781254
Image_08 0.611096 0.758609
Image_09 0.605927 0.754613
Image_10 0.645838 0.784814
Image_11 0.641263 0.781426
Image_12 0.655914 0.792207
Image_13 0.652899 0.790005
Image_14 0.666862 0.800140
Image_15 0.651382 0.788893
Image_16 0.685360 0.813310
Image_17 0.621204 0.766349
Image_18 0.670443 0.802713
Image_19 0.706089 0.827729
 Average 0.650459 0.787777
